# Reranking

Что, если мы хотим ещё больше повысить точность выдачи? Здесь на помощь приходят rerankers (переоценщики) — специальные модели для повторного ранжирования найденных документов.

Разберём, что такое cross-encoder модели, в чём их отличие от обычного векторного поиска (bi-encoder), и почему дополнительный этап повторного ранжирования заметно повышает качество выдачи в RAG-системах.

Rerankers — это модели, которые применяются к уже отобранным документам для их повторного ранжирования на основе более сложных критериев. Представьте, что первичный поиск (BM25 + векторный) — это широкая сеть, которая захватывает много потенциально релевантных документов. Reranker же — это эксперт, который внимательно изучает каждый документ и принимает финальное решение о его релевантности.

Основное преимущество rerankers заключается в том, что они могут учитывать более сложные зависимости между запросом и документом. Например, векторный поиск может дать высокий балл документу просто потому, что он на похожую тему, но не содержит ответа на конкретный вопрос. Reranker способен это исправить.


## Simple Reranker

In [ ]:
from sentence_transformers import CrossEncoder

class SimpleReranker:
    def __init__(self, model_name):
        """
        Args:
            model_name: Название модели cross-encoder
        """
        self.model = CrossEncoder(model_name)
    
    def rerank(self, query, documents, top_n=3):
        """      
        Args:
            query: Поисковый запрос
            documents: Список документов для переоценки
            top_n: Сколько документов возвращать
        Returns:
            Список топ-N документов после rerank
        """

        if not documents:
            return []
        
        # Подготавливаем пары (query, document)
        pairs = [[query, doc.page_content] for doc in documents]
        
        # Получаем скоры релевантности
        scores = self.model.predict(pairs)
        
        # Сортируем документы по скору
        doc_score_pairs = list(zip(documents, scores))
        doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
        
        # Отбираем топ-N документов (score записываем в метаданные)
        result = []
        for doc, score in doc_score_pairs[:top_n]:
            doc.metadata = doc.metadata or {}
            doc.metadata["rerank_score"] = float(score)
            result.append(doc)
        
        return result

Примеры популярных моделей (больше моделей на сайте Hugging Face по тегу Text Reranking):
- Alibaba-NLP/gte-multilingual-reranker-base — быстрая и мощная модель для мультиязычных задач, поддержка 70+
языков. Отличный баланс скорости и качества.
- BAAI/bge-reranker-base — универсальная модель с хорошим балансом скорости и качества. Поддержка 70+ языков.
- BAAI/bge-reranker-large — более точная версия, требует больше ресурсов. Лучшее качество при увеличенной латентности.
- cross-encoder/ms-marco-MiniLM-L-6-v2 — быстрая и лёгкая модель для английского языка. Хороший выбор для начала
экспериментов.
- cross-encoder/ms-marco-electra-base - более мощная модель для английского языка.
- Qwen/Qwen3-Reranker-0. 6B — компактная модель от Qwen, поддержка 100+ языков.
- Qwen/Qwen3-Reranker-4B , Qwen/Qwen3-Reranker-8B - более мощные версии Qwen3. Модель 8В занимает Nº1 в МТЕВ
multilingual leaderboard по качеству reranking.

In [1]:
# %%
import time

# Импорты стандартных инструментов для гибридного поиска и реранкинга
from langchain_classic.retrievers import (
    ContextualCompressionRetriever,
    EnsembleRetriever,
)
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS

# %%
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# Создаём тестовые документы
docs = [
    Document(
        page_content="Python - язык программирования для машинного обучения",
        metadata={"source": "doc_0", "category": "programming"},
    ),
    Document(
        page_content="Машинное обучение использует Python и математику",
        metadata={"source": "doc_1", "category": "ML"},
    ),
    Document(
        page_content="JavaScript работает в браузере",
        metadata={"source": "doc_2", "category": "web"},
    ),
    Document(
        page_content="Deep learning - подраздел машинного обучения",
        metadata={"source": "doc_3", "category": "ML"},
    ),
    Document(
        page_content="React - библиотека для JavaScript",
        metadata={"source": "doc_4", "category": "web"},
    ),
    Document(
        page_content="PyTorch - фреймворк для глубокого обучения",
        metadata={"source": "doc_5", "category": "ML"},
    ),
    Document(
        page_content="Node.js - среда выполнения JavaScript",
        metadata={"source": "doc_6", "category": "web"},
    ),
    Document(
        page_content="Нейронные сети в deep learning",
        metadata={"source": "doc_7", "category": "ML"},
    ),
]

# 1. Инициализируем компоненты
emb = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
vector_store = FAISS.from_documents(docs, emb)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 5})
bm25_retriever = BM25Retriever.from_documents(docs, k=5)

# 2. Создаем гибридный ретривер (Stage 1: Retrieval)
# ВАЖНО: EnsembleRetriever объединяет результаты, но НЕ реранжирует их Cross-Encoder-ом
ensemble_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    weights=[0.5, 0.5],
)

# 3. Создаем реранкер (Stage 2: Reranking)
# Используем специализированную модель-реранкер
cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",
)
compressor = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

# 4. Оборачиваем всё в ContextualCompressionRetriever
final_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
)

# %% ТЕСТИРУЕМ С ЗАМЕРОМ ВРЕМЕНИ
start_time = time.perf_counter()
results = final_retriever.invoke("расскажи про машинное обучение")
end_time = time.perf_counter()

latency = end_time - start_time

print(f"=== Гибридный поиск завершен за {latency:.4f} сек ===")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print(f"   Источник: {doc.metadata.get('source')}\n")


=== Гибридный поиск завершен за 0.1741 сек ===
1. Машинное обучение использует Python и математику
   Источник: doc_1

2. Deep learning - подраздел машинного обучения
   Источник: doc_3

3. Python - язык программирования для машинного обучения
   Источник: doc_0

4. PyTorch - фреймворк для глубокого обучения
   Источник: doc_5

5. Нейронные сети в deep learning
   Источник: doc_7

